# DeepSeek R1 Evaluation - With Reasoning Token Logging

This notebook evaluates DeepSeek R1 on the OI Benchmark and captures reasoning content.

**Install:** `pip install openai`

**Key Features:**
- No reasoning_effort parameter - always does full reasoning
- Recommended max_tokens: 8192 (quality degrades above this)

**Models:**
- `deepseek-reasoner` - Reasoning model (R1)
- `deepseek-chat` - Non-reasoning model

In [1]:
from openai import OpenAI

api_key = "sk-0a66f95492fe4706a536e3a16bc2a2c6"  # ← Your DeepSeek API key

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

In [2]:
import pandas as pd

df = pd.read_csv("OI_Benchmark.csv")  # ← Update path to your questions file
print(f"Loaded {len(df)} questions")
df.head()

Loaded 1010 questions


,question_ID,compound.name_1,SMILES_1,compound.name_2,SMILES_2,OPTIONS,question_category,prompt.1,prompt.2,answer,other_info
0,5e8d4af0-4df1-4fcd-b7c7-b7fe3bf1b630,"2-phenylethyl acetate, ethyl 2-methylbutanoate...","CC(=O)OCCC1=CC=CC=C1, CCC(C)C(=O)OCC, CCCCC1CC...",NaN,NaN,apple;mango;chocolate;peanut,smell_identification,"Given the molecules CC(=O)OCCC1=CC=CC=C1, CCC(...","Given the molecules 2-phenylethyl acetate, eth...",mango,NaN
1,b9df1df5-bbe7-4069-8a5f-976c699d87ad,"benzyl alcohol, benzaldehyde, phenylacetic aci...","C1=CC=C(C=C1)CO, C1=CC=C(C=C1)C=O, C1=CC=C(C=C...",NaN,NaN,peanut;fish;apple;walnut,smell_identification,"Given the molecules C1=CC=C(C=C1)CO, C1=CC=C(C...","Given the molecules benzyl alcohol, benzaldehy...",peanut,NaN
2,b9881c90-d7fa-4b77-823f-45279cbd594e,"(E,E,Z)-2,4,6-nonatrienal, 5-methyl-2-(E)-hept...","CC/C=C\C=C/C=C\C=O, CCC(C)C(=O)/C=C/C, CCC(C)C...",NaN,NaN,hazelnut;tomato;peanut;mango,smell_identification,"Given the molecules CC/C=C\C=C/C=C\C=O, CCC(C)...","Given the molecules (E,E,Z)-2,4,6-nonatrienal,...",hazelnut,NaN
3,a1ce026c-eb4d-49f5-ad0f-ed952ea353ec,"phenylacetic acid, 4-methylphenol (p-cresol), ...","C1=CC=C(C=C1)CC(=O)O, CC1=CC=C(C=C1)O, CCCC(=O...",NaN,NaN,onion;peach;cheese;tomato,smell_identification,"Given the molecules C1=CC=C(C=C1)CC(=O)O, CC1=...","Given the molecules phenylacetic acid, 4-methy...",tomato,NaN
4,aeab9337-d60c-4318-b6b4-5ae7695773e7,"methyl 2-methylbutanoate, ethyl 2-methylbutano...","CCC(C)C(=O)OC, CCC(C)C(=O)OCC, CCCC(=O)OCC, CC...",NaN,NaN,whisky;parsley;apple;melon,smell_identification,"Given the molecules CCC(C)C(=O)OC, CCC(C)C(=O)...","Given the molecules methyl 2-methylbutanoate, ...",apple,NaN


In [ ]:
import time, threading
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import APIError, RateLimitError, APITimeoutError

# ============ CONFIGURATION ============
# Model options:
# - "deepseek-reasoner" (R1 with reasoning - recommended)
# - "deepseek-chat" (non-reasoning)

model_name = "deepseek-reasoner"

# Max tokens: 8192 recommended for best quality
# Can go up to 32768 but quality may degrade
MAX_TOKENS = 16384

CSV_PATH = f"{model_name.replace('-', '_')}_OI_Benchmark.csv"

MAX_WORKERS = 4
RPM_LIMIT = 30
SAVE_EVERY = 5

# ============ COLUMNS FOR LOGGING ============
if 'answer_to_prompt_1' not in df.columns:
    df['answer_to_prompt_1'] = None
if 'answer_to_prompt_2' not in df.columns:
    df['answer_to_prompt_2'] = None
if 'input_tokens_1' not in df.columns:
    df['input_tokens_1'] = None
if 'input_tokens_2' not in df.columns:
    df['input_tokens_2'] = None
if 'output_tokens_1' not in df.columns:
    df['output_tokens_1'] = None
if 'output_tokens_2' not in df.columns:
    df['output_tokens_2'] = None
# DeepSeek exposes full reasoning content - we can count tokens
if 'reasoning_tokens_1' not in df.columns:
    df['reasoning_tokens_1'] = None
if 'reasoning_tokens_2' not in df.columns:
    df['reasoning_tokens_2'] = None

df.reset_index(drop=True, inplace=True)

# ============ RATE LIMITER ============
class RPMLimiter:
    def __init__(self, rpm):
        self.rpm = rpm
        self.calls = deque()
        self.lock = threading.Lock()

    def wait(self):
        while True:
            with self.lock:
                now = time.time()
                while self.calls and now - self.calls[0] >= 60:
                    self.calls.popleft()
                if len(self.calls) < self.rpm:
                    self.calls.append(now)
                    return
            time.sleep(0.05)

rpm_limiter = RPMLimiter(RPM_LIMIT)
save_lock = threading.Lock()

def save():
    with save_lock:
        df.to_csv(CSV_PATH, index=False)

# ============ API CALL - RETURNS ANSWER + TOKEN USAGE ============
def ask_until_success(prompt):
    """Returns: (answer_text, input_tokens, output_tokens, reasoning_tokens)"""
    while True:
        try:
            rpm_limiter.wait()
            
            response = client.chat.completions.create(
                model=model_name,
                max_tokens=MAX_TOKENS,
                messages=[{"role": "user", "content": prompt}]
            )
            
            # Get the final answer (content)
            text = ""
            if response.choices:
                message = response.choices[0].message
                text = (message.content or "").strip()
            
            # Extract token usage
            input_tokens = 0
            output_tokens = 0
            reasoning_tokens = 0
            
            if hasattr(response, 'usage') and response.usage:
                input_tokens = getattr(response.usage, 'prompt_tokens', 0) or 0
                output_tokens = getattr(response.usage, 'completion_tokens', 0) or 0
            
            # DeepSeek exposes reasoning_content - estimate token count
            # (roughly 1 token per 4 characters, or use word count * 1.3)
            if response.choices:
                message = response.choices[0].message
                reasoning_content = getattr(message, 'reasoning_content', None)
                if reasoning_content:
                    # Estimate reasoning tokens (words * 1.3 is a rough approximation)
                    reasoning_tokens = int(len(reasoning_content.split()) * 1.3)
            
            if text:
                return text, input_tokens, output_tokens, reasoning_tokens
            
            print("⛔ Empty response — retrying in 4s…")
            time.sleep(4)

        except RateLimitError as e:
            print(f"⚠️ Rate limit — retrying in 15s: {e}")
            time.sleep(15)
        except (APIError, APITimeoutError) as e:
            print(f"⚠️ API issue — retrying in 4s: {e}")
            time.sleep(4)
        except Exception as e:
            print(f"❌ Error — retrying in 4s: {e}")
            time.sleep(4)

# ============ WORKER ============
row_lock = threading.Lock()
progress = {"count": 0}

def process_row(i):
    if pd.notna(df.at[i, 'answer_to_prompt_1']) and pd.notna(df.at[i, 'answer_to_prompt_2']):
        return

    p1 = str(df.at[i, 'prompt.1'])
    p2 = str(df.at[i, 'prompt.2'])
    opt = str(df.at[i, 'OPTIONS'])

    if "OPTIONS" in p1: p1 = p1.replace("OPTIONS", opt)
    if "OPTIONS" in p2: p2 = p2.replace("OPTIONS", opt)

    a1, in1, out1, reason1 = ask_until_success(p1)
    a2, in2, out2, reason2 = ask_until_success(p2)

    with row_lock:
        df.at[i, 'answer_to_prompt_1'] = a1
        df.at[i, 'answer_to_prompt_2'] = a2
        df.at[i, 'input_tokens_1'] = in1
        df.at[i, 'input_tokens_2'] = in2
        df.at[i, 'output_tokens_1'] = out1
        df.at[i, 'output_tokens_2'] = out2
        df.at[i, 'reasoning_tokens_1'] = reason1
        df.at[i, 'reasoning_tokens_2'] = reason2
        progress["count"] += 1

        if progress["count"] % SAVE_EVERY == 0:
            save()
            print(f"💾 Saved @ {progress['count']} rows | Reasoning: ~{reason1}, ~{reason2} tokens")

# ============ RUN ============
todo = [i for i in range(len(df))
        if pd.isna(df.at[i, 'answer_to_prompt_1']) or pd.isna(df.at[i, 'answer_to_prompt_2'])]

print(f"🚀 {len(todo)} rows to process")
print(f"🤖 Model: {model_name}")
print(f"📊 Max tokens: {MAX_TOKENS}")
print(f"📁 Output: {CSV_PATH}")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_row, i) for i in todo]
    for idx, _ in enumerate(as_completed(futures), 1):
        if idx % 10 == 0:
            print(f"Progress: {idx}/{len(todo)}")

save()

# Final verification
missing = df[df['answer_to_prompt_1'].isna() | df['answer_to_prompt_2'].isna()].index.tolist()
while missing:
    print(f"🔄 Fixing {len(missing)} missing...")
    for i in missing:
        process_row(i)
        save()
    missing = df[df['answer_to_prompt_1'].isna() | df['answer_to_prompt_2'].isna()].index.tolist()

save()
print(f"✅ COMPLETE — Saved to {CSV_PATH}")

🚀 1010 rows to process
🤖 Model: deepseek-reasoner
📊 Max tokens: 16384
📁 Output: deepseek_reasoner_OI_Benchmark.csv


In [ ]:
# ============ SUMMARY STATISTICS ============
result = pd.read_csv(CSV_PATH)

print("\n" + "="*50)
print("📊 TOKEN USAGE SUMMARY")
print("="*50)

print(f"\nPrompt 1:")
print(f"  Mean input tokens:     {result['input_tokens_1'].mean():.1f}")
print(f"  Mean output tokens:    {result['output_tokens_1'].mean():.1f}")
print(f"  Mean reasoning tokens: {result['reasoning_tokens_1'].mean():.1f} (estimated)")

print(f"\nPrompt 2:")
print(f"  Mean input tokens:     {result['input_tokens_2'].mean():.1f}")
print(f"  Mean output tokens:    {result['output_tokens_2'].mean():.1f}")
print(f"  Mean reasoning tokens: {result['reasoning_tokens_2'].mean():.1f} (estimated)")

total_input = result['input_tokens_1'].sum() + result['input_tokens_2'].sum()
total_output = result['output_tokens_1'].sum() + result['output_tokens_2'].sum()
total_reasoning = result['reasoning_tokens_1'].sum() + result['reasoning_tokens_2'].sum()

print(f"\nTOTAL TOKENS:")
print(f"  Input:     {total_input:,}")
print(f"  Output:    {total_output:,}")
print(f"  Reasoning: {total_reasoning:,} (estimated from content length)")